In [1]:
import pandas as pd

In [5]:
# prop = 'mbj_bandgap'    
prop = "Tc_supercon"

df_text = pd.read_csv("./robo_0_75993.csv", index_col = 0)

df_train = pd.read_csv(f"/data/yll6162/alignntl_dft_3d/tl_dataset/dataset_alignn_matbert-base-cased_robo_prop_{prop}_train.csv", index_col = 0)
df_val = pd.read_csv(f"/data/yll6162/alignntl_dft_3d/tl_dataset/dataset_alignn_matbert-base-cased_robo_prop_{prop}_val.csv", index_col = 0)
df_test = pd.read_csv(f"/data/yll6162/alignntl_dft_3d/tl_dataset/dataset_alignn_matbert-base-cased_robo_prop_{prop}_test.csv", index_col = 0)
df_train = df_train.iloc[:, 768*2:]
df_test = df_test.iloc[:, 768*2:]
df_val = df_val.iloc[:, 768*2:]
df_text['files'] = df_text['jid'] + '.vasp'
df_train = df_train.merge(df_text, left_on = 'ids', right_on = 'files', how='left')
df_val = df_val.merge(df_text, left_on = 'ids', right_on = 'files', how='left')
df_test = df_test.merge(df_text, left_on = 'ids', right_on = 'files', how='left')

In [6]:
from jarvis.db.figshare import data
from jarvis.core.atoms import Atoms
import os

# Load the entire dataset once at the beginning
print("Loading JARVIS dataset...")
dataset = data("dft_3d")  # Load once, use many times
print(f"Dataset loaded with {len(dataset)} entries")

# Create a lookup dictionary for faster access
jarvis_dict = {entry['jid']: entry for entry in dataset}
print("Created lookup dictionary")

data_path = f"./data/{prop}/"
train_path = data_path + "train"
test_path = data_path + "test"
# prop = "mbj_bandgap"

# Create directories if they don't exist
os.makedirs(f"{train_path}/text_data", exist_ok=True)
os.makedirs(f"{train_path}/bulk_data", exist_ok=True)
os.makedirs(f"{test_path}/text_data", exist_ok=True)
os.makedirs(f"{test_path}/bulk_data", exist_ok=True)

# Process training data
print("Processing training data...")
for i, (_, row) in enumerate(df_train.iterrows()):
    jid = row['jid']
    robo_text = row['text']
    
    # Write text data
    with open(f"{train_path}/text_data/{jid}.txt", "w", encoding="utf-8") as f:
        f.write(str(robo_text))
    
    # Get data from pre-loaded dictionary (much faster!)
    if jid in jarvis_dict:
        rec = jarvis_dict[jid]
        atoms = Atoms.from_dict(rec["atoms"])
        # Write CIF and remove first line
        temp_cif_path = f"{train_path}/bulk_data/{jid}_temp.cif"
        final_cif_path = f"{train_path}/bulk_data/{jid}.cif"
        
        atoms.write_cif(temp_cif_path)
        
        # Read, remove first line, and rewrite
        with open(temp_cif_path, 'r') as f:
            lines = f.readlines()
        
        with open(final_cif_path, 'w') as f:
            f.writelines(lines[1:])  # Skip first line
        
        # Clean up temporary file
        os.remove(temp_cif_path)
    else:
        print(f"Warning: JID {jid} not found in dataset")
    
    if (i + 1) % 100 == 0:  # Progress indicator
        print(f"Processed {i + 1}/{len(df_train)} training samples")

# Save training targets
df_train[['jid', prop]].to_csv(f"{train_path}/targets_{prop}.csv", header=False, index=False)

# Process test data
print("Processing test data...")
for i, (_, row) in enumerate(df_test.iterrows()):
    jid = row['jid']
    robo_text = row['text']
    
    # Write text data
    with open(f"{test_path}/text_data/{jid}.txt", "w", encoding="utf-8") as f:
        f.write(str(robo_text))
    
    # Get data from pre-loaded dictionary
    if jid in jarvis_dict:
        rec = jarvis_dict[jid]
        atoms = Atoms.from_dict(rec["atoms"])
        # Write CIF and remove first line
        temp_cif_path = f"{test_path}/bulk_data/{jid}_temp.cif"
        final_cif_path = f"{test_path}/bulk_data/{jid}.cif"
        
        atoms.write_cif(temp_cif_path)
        
        # Read, remove first line, and rewrite
        with open(temp_cif_path, 'r') as f:
            lines = f.readlines()
        
        with open(final_cif_path, 'w') as f:
            f.writelines(lines[1:])  # Skip first line
        
        # Clean up temporary file
        os.remove(temp_cif_path)
    else:
        print(f"Warning: JID {jid} not found in dataset")
    
    if (i + 1) % 100 == 0:  # Progress indicator
        print(f"Processed {i + 1}/{len(df_test)} test samples")

# Save test targets (fixed the bug - was saving to train_path)
df_test[['jid', prop]].to_csv(f"{test_path}/targets_{prop}.csv", header=False, index=False)

print("Processing completed!")

Loading JARVIS dataset...
Obtaining 3D dataset 76k ...
Reference:https://www.nature.com/articles/s41524-020-00440-1
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.
Dataset loaded with 75993 entries
Created lookup dictionary
Processing training data...


/scratch/yll6162/miniconda3/envs/matmm/lib/python3.11/site-packages/jarvis/analysis/structure/spacegroup.py:230: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  return self._dataset["international"]


Processed 100/842 training samples
Processed 200/842 training samples
Processed 300/842 training samples
Processed 400/842 training samples
Processed 500/842 training samples
Processed 600/842 training samples
Processed 700/842 training samples
Processed 800/842 training samples
Processing test data...


/scratch/yll6162/miniconda3/envs/matmm/lib/python3.11/site-packages/jarvis/analysis/structure/spacegroup.py:230: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  return self._dataset["international"]


Processed 100/106 test samples
Processing completed!
